# CloudPath AI – AI-Based Cloud Movement Tracking and Rainfall Prediction System
## Phase 2: Machine Learning Model Development & Evaluation

**Dataset**: `bengaluru_processed.csv` (~96,431 hourly records, 2009–2020)  
**Author**: B.Tech Final Year Project  
**Scope**:
1. Chronological (time-series aware) data splitting (80% train / 20% test)
2. Binary Classification: Next-hour rainfall occurrence prediction (`rain_status`)
3. Regression: Next-hour precipitation quantity prediction (`next_hour_rainfall` in mm)
4. Time-series cross-validation hyperparameter tuning
5. Comprehensive performance evaluation and model diagnostics
6. Serialization of complete deployment pipelines (Joblib)

### 1. Environment Setup & Library Imports

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'savefig.bbox': 'tight'})

print('All libraries imported successfully!')

### 2. Dataset Understanding & Leakage Verification
We load `bengaluru_processed.csv` generated during Phase 1 preprocessing.

In [ ]:
CSV_PATH = 'bengaluru_processed.csv'
df = pd.read_csv(CSV_PATH)
df['date_time'] = pd.to_datetime(df['date_time'])
print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date Range: {df["date_time"].min()} to {df["date_time"].max()}')

# Map season to numeric ordinal values for modeling
season_map = {'Winter': 0, 'Pre-Monsoon': 1, 'Monsoon': 2, 'Post-Monsoon': 3}
df['season_enc'] = df['season'].map(season_map)

FEATURE_COLS = [
    'maxtempC', 'mintempC', 'sunHour', 'uvIndex', 'moon_illumination',
    'DewPointC', 'FeelsLikeC', 'HeatIndexC', 'WindChillC', 'WindGustKmph',
    'cloudcover', 'humidity', 'precipMM', 'pressure', 'tempC',
    'visibility', 'winddirDegree', 'windspeedKmph',
    'hour', 'day', 'month', 'year', 'day_of_week', 'week_of_year', 'season_enc'
]

TARGET_CLF = 'rain_status'
TARGET_REG = 'next_hour_rainfall'

# Data leakage check
assert TARGET_CLF not in FEATURE_COLS
assert TARGET_REG not in FEATURE_COLS
print(f'Verified: {len(FEATURE_COLS)} features. No targets present in feature set.')

### 3. Chronological Train-Test Split (80% Train / 20% Test)
To preserve temporal dynamics, **no random shuffling** is performed.

In [ ]:
split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f'Training Set : {len(train_df):,} rows ({train_df["date_time"].min().date()} to {train_df["date_time"].max().date()})')
print(f'Testing Set  : {len(test_df):,} rows ({test_df["date_time"].min().date()} to {test_df["date_time"].max().date()})')

X_train = train_df[FEATURE_COLS].values
X_test  = test_df[FEATURE_COLS].values
y_clf_train = train_df[TARGET_CLF].values
y_clf_test  = test_df[TARGET_CLF].values
y_reg_train = train_df[TARGET_REG].values
y_reg_test  = test_df[TARGET_REG].values

# Fit scaler strictly on training set
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Feature standard scaling applied using training parameters.')

### 4. Classification Model Training & Comparison (`rain_status`)
We compare Logistic Regression, Decision Tree, Random Forest, and XGBoost with class-balancing.

In [ ]:
clf_models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=10, min_samples_leaf=50, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', max_depth=15, min_samples_leaf=30, n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, scale_pos_weight=(y_clf_train==0).sum()/(y_clf_train==1).sum(), eval_metric='logloss', random_state=42, n_jobs=-1)
}

clf_metrics = {}
for name, model in clf_models.items():
    model.fit(X_train_sc, y_clf_train)
    preds = model.predict(X_test_sc)
    probs = model.predict_proba(X_test_sc)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc  = accuracy_score(y_clf_test, preds)
    prec = precision_score(y_clf_test, preds, zero_division=0)
    rec  = recall_score(y_clf_test, preds, zero_division=0)
    f1   = f1_score(y_clf_test, preds, zero_division=0)
    auc  = roc_auc_score(y_clf_test, probs) if probs is not None else np.nan
    
    clf_metrics[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': auc}

pd.DataFrame(clf_metrics).T.round(4)

### 5. Regression Model Training & Comparison (`next_hour_rainfall`)
Predicting actual precipitation depth in mm. Notice how zero-inflation influences MSE/MAE.

In [ ]:
reg_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=30, n_jobs=-1, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, objective='reg:squarederror', random_state=42, n_jobs=-1)
}

reg_metrics = {}
for name, model in reg_models.items():
    model.fit(X_train_sc, y_reg_train)
    preds = np.clip(model.predict(X_test_sc), 0, None)
    
    mae  = mean_absolute_error(y_reg_test, preds)
    mse  = mean_squared_error(y_reg_test, preds)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_reg_test, preds)
    
    reg_metrics[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

pd.DataFrame(reg_metrics).T.round(4)

### 6. Time-Series Aware Hyperparameter Tuning
Using `TimeSeriesSplit` cross-validation to tune hyperparameters without look-ahead bias.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

# Tuned XGBoost Classifier
tuned_clf = XGBClassifier(
    n_estimators=400, max_depth=7, learning_rate=0.05,
    scale_pos_weight=(y_clf_train==0).sum()/(y_clf_train==1).sum(),
    eval_metric='logloss', random_state=42, n_jobs=-1
)
tuned_clf.fit(X_train_sc, y_clf_train)
clf_preds = tuned_clf.predict(X_test_sc)
print('Tuned Classifier F1:', round(f1_score(y_clf_test, clf_preds), 4))

# Tuned XGBoost Regressor
tuned_reg = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.08,
    objective='reg:squarederror', random_state=42, n_jobs=-1
)
tuned_reg.fit(X_train_sc, y_reg_train)
reg_preds = np.clip(tuned_reg.predict(X_test_sc), 0, None)
print('Tuned Regressor RMSE:', round(np.sqrt(mean_squared_error(y_reg_test, reg_preds)), 4))

### 7. Model Diagnostic Visualizations
Confusion matrix, ROC curves, actual vs. predicted rainfall, and feature importances.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_clf_test, clf_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues', ax=axes[0],
            xticklabels=['No Rain', 'Rain'], yticklabels=['No Rain', 'Rain'])
axes[0].set_title('Normalized Confusion Matrix (%)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Top Feature Importances
importances = pd.Series(tuned_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(10)
importances.plot(kind='barh', ax=axes[1], color='#1976D2')
axes[1].set_title('Top 10 Feature Importances (Classifier)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 8. Serialization & Autonomous Pipeline Verification
Verify that saved pipelines can accept new, unscaled weather records directly.

In [ ]:
# Load saved pipeline
clf_pipe = joblib.load('rainfall_classifier.pkl')
reg_pipe = joblib.load('rainfall_regressor.pkl')

# Sample new unseen observation
sample_raw = test_df[FEATURE_COLS].iloc[[10]]
sample_scaled = clf_pipe['scaler'].transform(sample_raw.values)

pred_prob = clf_pipe['model'].predict_proba(sample_scaled)[0, 1]
pred_class = clf_pipe['model'].predict(sample_scaled)[0]
pred_amount = np.clip(reg_pipe['model'].predict(sample_scaled)[0], 0, None)

print(f'Test Sample Weather Record:')
print(f'  Current Precip: {sample_raw["precipMM"].values[0]} mm | Humidity: {sample_raw["humidity"].values[0]}% | Cloudcover: {sample_raw["cloudcover"].values[0]}%')
print(f'Pipeline Predictions for Next Hour:')
print(f'  Rain Occurrence Probability : {pred_prob*100:.1f}% -> Status: {pred_class} ({"Rain" if pred_class==1 else "No Rain"})')
print(f'  Predicted Rainfall Quantity : {pred_amount:.2f} mm')
print(f'  Actual Observed Next Hour   : {test_df["next_hour_rainfall"].iloc[10]:.2f} mm')